# Series temporales — pronóstico de ventas
Compara naive estacional, SARIMA, SARIMAX y Prophet con el mismo walk-forward.
`data/processed/master_pedidos.csv` → `artifacts/timeseries/`

In [ ]:
import sys, json, warnings, logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
for n in ("cmdstanpy", "prophet"):
    logging.getLogger(n).setLevel(logging.CRITICAL)

ROOT = Path.cwd().parents[1] if Path.cwd().name == "05_timeseries" else Path.cwd()
sys.path.insert(0, str(ROOT))
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams.update({"figure.figsize": (12, 4), "figure.dpi": 110, "axes.titleweight": "bold"})

from src.models.timeseries import (build_series, calendario, comparar, detalle_por_pliegue,
                                   evaluar, metricas, naive_estacional, pronostico_final,
                                   walk_forward_splits)
print("ROOT:", ROOT.resolve())

## 1. La serie

In [ ]:
serie = build_series()
print(f"{len(serie)} días | {serie.index[0].date()} a {serie.index[-1].date()}")
serie.describe().round(0).to_frame("ventas_BRL")

## 2. Por qué el recorte
El rango completo arranca en el lanzamiento de la plataforma y termina en un
corte de extracción; ninguno de los dos extremos es demanda real.

In [ ]:
crudo = build_series(inicio="2017-01-05", fin="2018-08-29")
fig, ax = plt.subplots()
ax.plot(crudo.index, crudo / 1000, lw=.8, color="#c0504d", label="descartado")
ax.plot(serie.index, serie / 1000, lw=.8, color="#4f81bd", label="serie usada")
ax.set(title="Ventas diarias (miles de BRL)", ylabel="miles BRL")
ax.legend(); plt.tight_layout(); plt.show()

extremos = pd.DataFrame({
    "tramo": ["arranque descartado", "serie usada", "cola descartada"],
    "mediana_BRL": [crudo.loc[:"2017-01-22"].median(), serie.median(), crudo.loc["2018-08-23":].median()],
})
extremos["% de la serie usada"] = (extremos.mediana_BRL / serie.median() * 100).round(1)
extremos

## 3. Estacionalidad semanal

In [ ]:
dow = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
patron = serie.groupby(serie.index.dayofweek).median().set_axis(dow)

fig, ax = plt.subplots(figsize=(8, 3.5))
patron.div(1000).plot.bar(ax=ax, color="#4f81bd")
ax.set(title="Mediana de ventas por día de la semana", ylabel="miles BRL", xlabel="")
ax.tick_params(axis="x", rotation=0); plt.tight_layout(); plt.show()

print(f"El sábado vende {(1 - patron.min() / patron.max()) * 100:.0f}% menos que el martes.")
print("Ese patrón es lo que justifica la estacionalidad s=7.")

## 4. Validación con origen móvil
Seis pliegues de 30 días. Cada uno entrena solo con el pasado: barajar el orden
filtraría el futuro hacia el pasado.

In [ ]:
for i, (tr, te) in enumerate(walk_forward_splits(serie), 1):
    print(f"pliegue {i}: train {len(tr):3} días → test {te.index[0].date()} a {te.index[-1].date()}")

## 5. Baseline: naive estacional
Repite el mismo día de la semana anterior. Si un modelo no le gana, no hay modelo.

In [ ]:
base = evaluar(serie, naive_estacional)
print(base.round(1).to_string(index=False))
print(f"\npromedio -> MAE {base.MAE.mean():,.0f} | sMAPE {base['sMAPE'].mean():.1f}%")

## 6. Comparación de los cuatro modelos
SARIMA y SARIMAX trabajan sobre `log1p` y eligen su orden por AIC usando solo el
train de cada pliegue.

In [ ]:
resumen = comparar(serie)
resumen.round(2)

In [ ]:
detalle = detalle_por_pliegue(serie)
detalle.pivot(index="pliegue", columns="modelo", values="MAE").round(0)

## 7. Por qué SARIMAX no mejora
Sus regresores de calendario valen cero en los seis pliegues de validación.

In [ ]:
filas = []
for i, (tr, te) in enumerate(walk_forward_splits(serie), 1):
    ex = calendario(te.index)
    filas.append({"pliegue": i, **{c: int(ex[c].sum()) for c in ex.columns}})
pd.DataFrame(filas)

In [ ]:
ex = calendario(serie.index)
print("días con evento en toda la serie:", {c: int(ex[c].sum()) for c in ex.columns})
print()
print("El dataset tiene un solo Black Friday (24-nov-2017), así que su efecto cae")
print("siempre en train o en test pero nunca de forma útil en ambos: con 19 meses")
print("no se puede estimar y validar a la vez.")

## 8. Modelo ganador
SARIMA es el único que supera al baseline.

In [ ]:
ganador = resumen.iloc[0]
print(f"{ganador.modelo}: MAE {ganador.MAE:,.0f} | sMAPE {ganador['sMAPE']:.1f}% | "
      f"{ganador['mejora_MAE_%']:.1f}% mejor que el baseline")

futuro, ajuste, (order, seasonal) = pronostico_final(serie)
print(f"orden elegido por AIC: SARIMA{order}{seasonal}  (AIC {ajuste.aic:.1f})")

In [ ]:
tr, te = list(walk_forward_splits(serie))[-1]
from src.models.timeseries import sarima
pred = sarima(tr, 30)

fig, ax = plt.subplots()
ax.plot(serie.loc["2018-05-01":].index, serie.loc["2018-05-01":] / 1000, lw=.9, color="#4f81bd", label="real")
ax.plot(pred.index, pred / 1000, lw=1.6, color="#c0504d", label="SARIMA (último pliegue)")
ax.plot(futuro.fecha, futuro.pronostico / 1000, lw=1.6, color="#4f9a4f", label="pronóstico 30 días")
ax.fill_between(futuro.fecha, futuro.inferior / 1000, futuro.superior / 1000, color="#4f9a4f", alpha=.15)
ax.set(title="Backtest y pronóstico", ylabel="miles BRL")
ax.legend(); plt.tight_layout(); plt.show()

## 9. KPI comprometido en el 1° avance

In [ ]:
sem_base, sem_mod = [], []
for tr, te in walk_forward_splits(serie):
    for pred_fn, acc in ((naive_estacional, sem_base), (sarima, sem_mod)):
        p = pred_fn(tr, len(te))
        r, q = te.resample("W").sum(), pd.Series(np.asarray(p), index=te.index).resample("W").sum()
        k = min(len(r), len(q))
        acc.append(metricas(r.to_numpy()[:k], q.to_numpy()[:k]))

print(f"MAPE semanal  baseline {pd.DataFrame(sem_base).MAPE.mean():.1f}%  "
      f"SARIMA {pd.DataFrame(sem_mod).MAPE.mean():.1f}%   (meta comprometida: <= 20%)")
print("\nEl baseline trivial ya cumple la meta, así que la vara real es superarlo.")

## 10. Artefactos

In [ ]:
from src.models.timeseries import generar_artefactos

manifest = generar_artefactos()
print(json.dumps({k: manifest[k] for k in ("status", "model_name", "version", "winner", "metrics")},
                 ensure_ascii=False, indent=2))

## Conclusiones
SARIMA(1,1,2)(1,1,1,7) sobre `log1p` gana: es el único de los cuatro que supera
al naive estacional, con 4.9% menos de MAE. Captura la estacionalidad semanal,
que es el patrón dominante de la serie.

SARIMAX no aporta porque sus regresores son cero en toda la validación, y
Prophet queda por debajo del baseline en las seis configuraciones probadas.

**Limitaciones:** 19 meses útiles no permiten estimar estacionalidad anual, solo
la semanal; un único Black Friday impide modelar eventos especiales; el dataset
termina en 2018, por lo que el pronóstico es una demostración metodológica y no
una proyección operativa.